# Mutual Information Evaluation

This notebook estimates the mutual information between the original and
anonymized embeddings using the MINDE estimator.

> **Important**
>
> This notebook requires the original MINDE repository and its dedicated
> Python environment.
>
> Repository:
> https://github.com/MustaphaBounoua/minde

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
MINDE_ROOT = PROJECT_ROOT / "minde"

if not MINDE_ROOT.exists():
    raise FileNotFoundError(
        "MINDE repository not found.\n"
        "Clone it into the project root as:\n"
        "project/minde/"
    )

sys.path.insert(0, str(MINDE_ROOT))

In [ ]:
import torch
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader

from src.libs.minde import MINDE
from src.scripts.helper import get_default_config

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {device}")

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"

INPUT_PATH = DATA_DIR / "case_embeddings.parquet"

ANON_DIR = DATA_DIR / "anonymized_outputs"

OUTPUT_DIR = DATA_DIR / "results"
OUTPUT_DIR.mkdir(exist_ok=True)

CLUSTER_K = [5, 10, 25, 50, 100]

In [ ]:
X = pd.read_parquet(INPUT_PATH).to_numpy(dtype=np.float32)
print(X.shape)

In [ ]:
scaler = StandardScaler()

X = scaler.fit_transform(X)

In [ ]:
class MINDataset(Dataset):

    def __init__(self, X, Y):

        self.X = torch.tensor(
            X,
            dtype=torch.float32,
        )

        self.Y = torch.tensor(
            Y,
            dtype=torch.float32,
        )

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return {
            "x": self.X[idx],
            "y": self.Y[idx],
        }

In [ ]:
args = get_default_config()

args.type = "c"
args.importance_sampling = True
args.arch = "mlp"
args.use_ema = True

args.max_epochs = 150
args.warmup_epochs = 5
args.test_epoch = 5

args.lr = 1e-3
args.bs = 512

args.accelerator = (
    "gpu"
    if torch.cuda.is_available()
    else "cpu"
)

args.devices = 1

args.out_dir = "./outputs"
args.benchmark = "embedding_mi"
args.seed = 42

args.mc_iter = 20

In [ ]:
def estimate_mi(
    original_embeddings,
    anonymized_embeddings,
):

    anonymized_embeddings = scaler.transform(
        anonymized_embeddings
    )

    dataset = MINDataset(
        original_embeddings,
        anonymized_embeddings,
    )

    loader = DataLoader(
        dataset,
        batch_size=args.bs,
        shuffle=True,
        num_workers=0,
    )

    model = MINDE(
        args,
        var_list={
            "x": original_embeddings.shape[1],
            "y": anonymized_embeddings.shape[1],
        },
        gt=None,
    )

    model.fit(
        loader,
        loader,
    )

    return model.compute_mi()

In [ ]:
results = []

algorithms = {
    "MDAV-C": "mdav_c_k{}.parquet",
    "RMDAV-M": "rmdav_m_k{}.parquet",
}

for algorithm, pattern in algorithms.items():

    for cluster_k in CLUSTER_K:

        print(
            f"{algorithm} | k={cluster_k}"
        )

        X_anon = pd.read_parquet(
            ANON_DIR / pattern.format(cluster_k)
        ).to_numpy(dtype=np.float32)

        mi, mi_sigma = estimate_mi(
            X,
            X_anon,
        )

        results.append(
            {
                "algorithm": algorithm,
                "cluster_k": cluster_k,
                "mi": mi,
                "mi_sigma": mi_sigma,
            }
        )

In [ ]:
results_df = pd.DataFrame(results)

results_df

In [ ]:
results_df.to_csv(
    OUTPUT_DIR / "mutual_information.csv",
    index=False,
)

print("Results saved successfully.")